# 망막 혈관 분할 (DRIVE) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 망막 혈관 (Retinal Vessel) 세그멘테이션
- **모달리티**: Fundus Photography (안저 촬영, RGB)
- **태스크**: Binary segmentation — 배경(0) / 혈관(1)
- **핵심 도전**: 모세혈관 등 극세 혈관 검출, 시신경 유두 주변 고밀도 혈관망, 촬영 밝기 편차

## 2. 모델
- **아키텍처**: IterNet (경량 CNN + 반복 정제)
- **사전학습**: 없음 (from scratch)
- **선택 이유**: 혈관 세그멘테이션 특화 경량 모델, 패치 기반 학습에 적합
- **출력**: 1채널 sigmoid → `to_2ch_logits()` 변환 후 손실 함수 적용
- **참고 모델**: PraNet (Res2Net50) 도 코드 포함 — 비교 가능

## 3. 데이터셋
- **이름**: DRIVE (Digital Retinal Images for Vessel Extraction)
- **규모**: 40장 (train 20 / test 20) — train에서 Val 20% 분리, Test GT 없음
- **입력 방식**: 패치 기반 — 전체 이미지에서 256×256 랜덤 패치 추출 (혈관 중심 샘플링 50%)
- **클래스 불균형**: BG:Vessel = **~10.7:1** (심각)
- **공식 분할**: 공식 test set에 vessel GT 없음 → **val set으로 최종 평가** (불가피)

## 4. 데이터 준비 (협업자용)
> Cell 0 실행 시 자동으로 데이터가 다운로드됨. 별도 준비 불필요.

**취득 방법 (자동)**:
```python
kagglehub.dataset_download("andrewmvd/drive-digital-retinal-images-for-vessel-extraction")
```

## 5. 전처리 및 도메인 특이점
- CLAHE 적용 (Green 채널, clipLimit=2.0, tileGridSize=8×8) — 혈관 대비 강조
- 3채널 입력: CLAHE 처리된 Green 채널을 3채널로 복제
- 패치 샘플링: 50% 확률로 혈관 픽셀 중심 패치 추출 → 혈관 클래스 노출 빈도 증가
- ImageNet 정규화 적용

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 20 |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 40 |

## 7. SoTA 참고 (2026년 3월 기준)
| 방법 | Dice | AUC | Sensitivity | Specificity | 출처 |
|------|------|-----|-------------|-------------|------|
| TransUNext (2024) | — | **0.9867** | 0.8208 | 0.9840 | Computers in Biology |
| GAN+UNet (2024) | 0.8215 | 0.9772 | 0.8301 | 0.9781 | IEEE JBHI |
| LU-RA Transformer (2025) | 0.7871 | — | — | — | Applied Intelligence |
| IterNet (원본) | ~0.779 | ~0.979 | — | — | AAAI'20 |
| U-Net baseline | ~0.730 | ~0.974 | — | — | 복수 논문 |

> ⚠️ DRIVE test set에는 vessel GT 없음 → val set(4장)으로 평가. 결과 해석 시 참고.
> 본 연구 목표: IterNet baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: Dice, Sensitivity, Specificity, AUC
> 결과 저장: `medical_data/results/Retinal_Image/`

In [ ]:
# === Cell 0: 환경 설정 ===
import os, sys, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import kagglehub
import glob
from tqdm import tqdm
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# custom_losses.py 경로 추가 (로컬)
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# DRIVE 데이터셋 로드 (kagglehub 캐시 활용)
path = kagglehub.dataset_download("andrewmvd/drive-digital-retinal-images-for-vessel-extraction")
DATA_DIR = path

all_tifs = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.tif"), recursive=True))
all_gifs = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.gif"), recursive=True))

# manual1만 사용 (각 이미지에 manual1/manual2 두 개의 어노테이션 존재)
train_img_paths  = sorted([p for p in all_tifs if 'training' in p.lower()])
train_mask_paths = sorted([p for p in all_gifs if 'training' in p.lower() and 'manual1' in p.lower()])
# DRIVE test set에는 vessel GT(1st_manual)가 없으므로 train에서 val split 사용
test_img_paths   = sorted([p for p in all_tifs if 'test' in p.lower()])
test_mask_paths  = []  # GT 없음 — 평가는 val set으로 대체

# Train / Val 분리
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    train_img_paths, train_mask_paths, test_size=0.2, random_state=42)

# --- 결과 저장 경로 ---
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/Retinal_Image'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Train {len(train_img_paths)} | Val {len(val_img_paths)} | Test {len(test_img_paths)} (GT 없음)")

In [ ]:
# === Cell 2: 유틸리티 함수 ===
# --- 1채널 → 2채널 대칭 logit 변환 (핵심 버그 수정) ---
# 기존: [zeros, p]  → 배경 logit 0 고정, 학습 불안정
# 수정: [-p, p]    → 대칭 logit, sigmoid(p)와 수학적으로 동일
def to_2ch_logits(p):
    return torch.cat([-p, p], dim=1)

# 검증 Dice 계산
def compute_val_dice(model, loader):
    model.eval()
    dice_sum = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            res   = preds[-1] if isinstance(preds, (list, tuple)) else preds
            res   = F.interpolate(res, size=masks.shape[1:], mode='bilinear', align_corners=True)
            prob  = torch.sigmoid(res).squeeze(1)
            pred  = (prob > 0.5).long()
            inter = (pred.float() * masks.float()).sum()
            union = pred.float().sum() + masks.float().sum()
            dice_sum += (2. * inter / (union + 1e-8)).item() if union > 0 else 1.0
    return dice_sum / len(loader)

print("유틸리티 함수 정의 완료")

In [ ]:
# === Cell 3: 모델 정의 ===
# PraNet (이미 /tmp/PraNet에 클론+패치 완료 전제)
pranet_lib = '/tmp/PraNet/lib'
if pranet_lib not in sys.path:
    sys.path.insert(0, pranet_lib)

# 미클론 시 자동 클론+패치
if not os.path.exists('/tmp/PraNet'):
    os.system('git clone https://github.com/DengPingFan/PraNet.git /tmp/PraNet')
    # 상대 import 패치
    target = '/tmp/PraNet/lib/PraNet_Res2Net.py'
    with open(target) as f: code = f.read()
    if 'from .Res2Net_v1b' in code:
        with open(target, 'w') as f: f.write(code.replace('from .Res2Net_v1b', 'from Res2Net_v1b'))
    # Res2Net 가중치 경로 패치
    import urllib.request
    wp = '/tmp/PraNet/models/res2net50_v1b_26w_4s-3cf99910.pth'
    os.makedirs('/tmp/PraNet/models', exist_ok=True)
    if not os.path.exists(wp):
        urllib.request.urlretrieve(
            'https://shanghuagao.oss-cn-beijing.aliyuncs.com/res2net/res2net50_v1b_26w_4s-3cf99910.pth', wp)
    r2n = '/tmp/PraNet/lib/Res2Net_v1b.py'
    with open(r2n) as f: code = f.read()
    hc = '/media/nercms/NERCMS/GepengJi/Medical_Seqmentation/CRANet/models/res2net50_v1b_26w_4s-3cf99910.pth'
    if hc in code:
        with open(r2n, 'w') as f: f.write(code.replace(hc, wp))

for k in [k for k in sys.modules if 'Res2Net' in k or 'PraNet_Res2Net' in k]:
    del sys.modules[k]
from PraNet_Res2Net import PraNet

# IterNet (BatchNorm 적용)
class IterNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.out  = nn.Conv2d(64, 1, 1)
        self.iter = nn.Sequential(
            nn.Conv2d(65, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 1, 1))
    def forward(self, x):
        feat = self.base(x)
        out1 = self.out(feat)
        return [out1, self.iter(torch.cat([feat, out1], dim=1))]

print("모델 정의 완료: IterNet, PraNet")

In [ ]:
# === Cell 4: 학습 함수 + Optuna ===
import traceback as _tb
os.environ['TQDM_DISABLE'] = '1'

def train_model(model_class, loss_name, alpha=1.0, gamma=2.0, epochs=25, lr=1e-4):
    model     = model_class().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)
    name      = f"{model_class.__name__}+{loss_name}"
    print(f"\n{'='*55}\n{name}  (Epochs={epochs})\n{'='*55}")

    history   = {'loss': [], 'val_dice': []}
    best_dice = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, masks in tqdm(train_loader, desc=f"Ep{epoch+1:02d}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss  = 0
            if isinstance(preds, (list, tuple)):
                for p in preds:
                    p    = F.interpolate(p, size=masks.shape[1:], mode='bilinear', align_corners=True)
                    loss += criterion(to_2ch_logits(p), masks)
            else:
                p    = F.interpolate(preds, size=masks.shape[1:], mode='bilinear', align_corners=True)
                loss = criterion(to_2ch_logits(p), masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_loss = train_loss / len(train_loader)
        val_dice = compute_val_dice(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f"Ep{epoch+1:02d} | Loss:{avg_loss:.4f} | ValDice:{val_dice:.4f}", end="")
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f'/tmp/best_{name}.pth')
            print("  ← Best!", end="")
        print()

    model.load_state_dict(torch.load(f'/tmp/best_{name}.pth', weights_only=True))
    print(f"최고 Val Dice: {best_dice:.4f}")
    return model, history, best_dice

# IterNet + 4가지 Loss 비교
all_results = {}
for loss_name in ['ce_dice', 'wce_dice', 'lwce_dice', 'plwce_dice']:
    m, h, b = train_model(IterNet, loss_name, epochs=25)
    all_results[f'IterNet+{loss_name}'] = {'model': m, 'history': h, 'best_dice': b}

print("\n[IterNet Loss 비교 요약]")
for k, v in all_results.items():
    print(f"  {k}: Best Val Dice = {v['best_dice']:.4f}")


# --- PLWCE+Focal: joint alpha+gamma Optuna 탐색 ---
import optuna as _optuna, json as _json
_optuna.logging.set_verbosity(_optuna.logging.WARNING)

ALPHA_LOW_PF  = 2.5;  ALPHA_HIGH_PF = 15.0
GAMMA_LOW_PF  = 0.5;  GAMMA_HIGH_PF = 5.0
N_TRIALS_PF   = 60
PROXY_EPOCHS_PF = 8

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW_PF, ALPHA_HIGH_PF)
    gamma = trial.suggest_float('gamma', GAMMA_LOW_PF, GAMMA_HIGH_PF)
    try:
        _, _, dice = train_model(IterNet, 'plwce_focal_dice',
                                 alpha=alpha, gamma=gamma, epochs=PROXY_EPOCHS_PF)
        return dice
    except Exception as e:
        print(f"Trial {trial.number} 실패: {e}")
        _tb.print_exc()
        return 0.0

N_ALPHA_GRID = 10  # 10x6=60 grid
N_GAMMA_GRID = 6
sampler_pf = _optuna.samplers.GridSampler({
    'alpha': np.linspace(ALPHA_LOW_PF, ALPHA_HIGH_PF, N_ALPHA_GRID).tolist(),
    'gamma': np.linspace(GAMMA_LOW_PF, GAMMA_HIGH_PF, N_GAMMA_GRID).tolist(),
})
study_pf = _optuna.create_study(
    direction='maximize', study_name='drive_plwce_focal',
    sampler=sampler_pf
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF, show_progress_bar=True)

best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f"\n[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  (Val Dice={study_pf.best_value:.4f})")

# 탐색 결과 시각화 (2D scatter)
trials_pf = [t for t in study_pf.trials if t.value is not None]
alphas_pf  = [t.params['alpha'] for t in trials_pf]
gammas_pf  = [t.params['gamma'] for t in trials_pf]
values_pf  = [t.value           for t in trials_pf]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(alphas_pf, gammas_pf, c=values_pf, cmap='viridis', s=60, alpha=0.8)
ax.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, marker='*', zorder=5,
           label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax, label='Val Dice')
ax.set_xlabel('alpha'); ax.set_ylabel('gamma')
ax.set_title('PLWCE+Focal Optuna 탐색 (DRIVE)'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'DRIVE_optuna_search_pf.png'), dpi=100)
plt.show()

# PLWCE+Focal 최종 학습
m_pf, h_pf, b_pf = train_model(IterNet, 'plwce_focal_dice',
                                 alpha=best_alpha_pf, gamma=best_gamma_pf, epochs=25)
all_results['IterNet+plwce_focal_dice'] = {'model': m_pf, 'history': h_pf, 'best_dice': b_pf}

# Optuna 결과 저장
optuna_save = {
    'plwce_focal': {
        'best_alpha': best_alpha_pf, 'best_gamma': best_gamma_pf,
        'best_proxy_dice': study_pf.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                    'gamma': t.params.get('gamma'), 'value': t.value}
                   for t in study_pf.trials if t.value is not None]
    }
}
with open(os.path.join(RESULTS_DIR, 'DRIVE_optuna_results_pf.json'), 'w') as f:
    _json.dump(optuna_save, f, indent=2, ensure_ascii=False)
print(f"Optuna 결과 저장: {os.path.join(RESULTS_DIR, 'DRIVE_optuna_results_pf.json')}")

In [ ]:
# === Cell 5: 시각화 ===
DOMAIN = 'DRIVE'
COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7', '#956CB4']

def fast_predict(model, img_path):
    model.eval()
    img     = cv2.imread(img_path)
    h0, w0  = img.shape[:2]
    clahe   = cv2.createCLAHE(clipLimit=2.0)
    green   = clahe.apply(img[:, :, 1])
    resized = cv2.resize(cv2.merge([green] * 3), (256, 256))
    tf      = A.Compose([A.Normalize(), ToTensorV2()])
    tensor  = tf(image=resized)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        preds = model(tensor)
        res   = preds[-1] if isinstance(preds, (list, tuple)) else preds
        res   = F.interpolate(res, size=(h0, w0), mode='bilinear', align_corners=True)
        return torch.sigmoid(res).squeeze().cpu().numpy()

best_key   = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_key]['model']
print(f"시각화 모델: {best_key}")

# --- 학습 곡선 (Train Loss + Val Dice) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_dice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=7); ax1.grid(True)
ax2.set_title('Val Dice (Vessel)'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=7); ax2.grid(True)

plt.suptitle('DRIVE — IterNet 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 곡선 저장: {RESULTS_DIR}/{DOMAIN}_training_curves.png')

# --- 예측 결과 시각화 (DRIVE test에 GT 없음 → val set 사용) ---
# 4열: Input Fundus / GT / Prob Map / Pred
vis_imgs  = val_img_paths[:4]
vis_masks = val_mask_paths[:4]

fig, axes = plt.subplots(len(vis_imgs), 4, figsize=(18, len(vis_imgs) * 4))
if len(vis_imgs) == 1:
    axes = axes[np.newaxis, :]

for row, (im_p, mk_p) in enumerate(zip(vis_imgs, vis_masks)):
    orig = cv2.cvtColor(cv2.imread(im_p), cv2.COLOR_BGR2RGB)
    gt   = np.array(Image.open(mk_p).convert('L')) / 255.0
    prob = fast_predict(best_model, im_p)
    prob_r = cv2.resize(prob, (gt.shape[1], gt.shape[0]))
    pred = (prob_r > 0.5).astype(np.uint8)

    axes[row, 0].imshow(orig);                    axes[row, 0].set_title('Input Fundus');  axes[row, 0].axis('off')
    axes[row, 1].imshow(gt,   cmap='gray');        axes[row, 1].set_title('Ground Truth');  axes[row, 1].axis('off')
    axes[row, 2].imshow(prob_r, cmap='jet', vmin=0, vmax=1); axes[row, 2].set_title('Vessel Prob Map'); axes[row, 2].axis('off')
    axes[row, 3].imshow(pred, cmap='gray');        axes[row, 3].set_title(f'Pred ({best_key[:12]})'); axes[row, 3].axis('off')

plt.suptitle(f'DRIVE — 예측 결과 시각화 ({best_key})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'예측 결과 저장: {RESULTS_DIR}/{DOMAIN}_prediction_vis.png')


In [ ]:
# === Cell 6: 평가 및 결과 저장 ===
import json, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix

def evaluate_model(model, img_paths, mask_paths):
    dices, sens_list, spec_list, aucs = [], [], [], []
    for im_p, mk_p in zip(img_paths, mask_paths):
        gt   = (np.array(Image.open(mk_p).convert('L')) > 127).astype(np.uint8)
        prob = fast_predict(model, im_p)
        prob = cv2.resize(prob, (gt.shape[1], gt.shape[0]))
        pred = (prob > 0.5).astype(np.uint8)
        try:
            aucs.append(roc_auc_score(gt.flatten(), prob.flatten()))
        except:
            aucs.append(0.5)
        tn, fp, fn, tp = confusion_matrix(gt.flatten(), pred.flatten(), labels=[0, 1]).ravel()
        dices.append((2. * tp) / (2. * tp + fp + fn + 1e-8))
        sens_list.append(tp / (tp + fn + 1e-8))
        spec_list.append(tn / (tn + fp + 1e-8))
    return {
        'Dice':        float(np.mean(dices)),
        'Sensitivity': float(np.mean(sens_list)),
        'Specificity': float(np.mean(spec_list)),
        'AUC':         float(np.mean(aucs)),
    }

# DRIVE test set에 vessel GT 없음 → val set으로 평가
print("\n[전체 모델 종합 평가 — Val Set (DRIVE test에 GT 없음)]")
print(f"{'Model':<32} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 60)

final_results = {}
for key, val in all_results.items():
    m = evaluate_model(val['model'], val_img_paths, val_mask_paths)
    # Extract loss_name from key (format: 'IterNet+loss_name')
    lname = key.replace('IterNet+', '') if 'IterNet+' in key else key
    final_results[key] = {
        'loss_name':     lname,
        'alpha':         best_alpha_pf if 'focal' in lname else 1.0,
        'best_val_dice': val['best_dice'],
        **m,
    }
    print(f"{key:<32} {m['Dice']:>7.4f} {m['Sensitivity']:>7.4f} {m['Specificity']:>7.4f} {m['AUC']:>7.4f}")

# --- 바차트 비교 ---
labels_     = list(final_results.keys())
metric_keys = ['Dice', 'Sensitivity', 'Specificity', 'AUC']

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mkey in zip(axes, metric_keys):
    scores = [final_results[lb][mkey] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('+')[-1][:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(mkey); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.4)
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red'); bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('DRIVE — Loss별 최종 평가 지표 비교 (Val Set)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'평가 차트 저장: {RESULTS_DIR}/{DOMAIN}_final_metrics.png')

# --- JSON 저장 ---
# class_counts: 전체 train 픽셀 기준
bg_px = int(sum((np.array(Image.open(p).convert('L')) <= 127).sum() for p in train_mask_paths))
fg_px = int(sum((np.array(Image.open(p).convert('L')) >  127).sum() for p in train_mask_paths))

save_data = {
    'domain':       'DRIVE Retinal Vessel Segmentation',
    'model':        'IterNet (from scratch)',
    'sota_ref':     {'TransUNext (2024)': {'AUC': 0.9867, 'Sensitivity': 0.8208},
                     'IterNet (AAAI20)':  {'Dice': 0.779,  'AUC': 0.979}},
    'num_classes':  2,
    'class_counts': {'BG': bg_px, 'Vessel': fg_px},
    'imbalance':    {'BG_Vessel': round(bg_px / max(fg_px, 1), 1)},
    'train_count':  len(train_img_paths),
    'val_count':    len(val_img_paths),
    'eval_set':     'Val Set (DRIVE test GT 없음)',
    'final_epochs': 25,
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items() if mk != 'model'}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['Dice']),
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {RESULTS_DIR}/{DOMAIN}_final_results.json')

# --- Excel 저장 ---
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss_Function':    label,
        'loss_name':        v['loss_name'],
        'alpha':            round(float(v['alpha']), 4),
        'Best_Val_Dice':    round(v['best_val_dice'], 4),
        'Test_Dice':        round(v['Dice'],        4),
        'Test_Sensitivity': round(v['Sensitivity'], 4),
        'Test_Specificity': round(v['Specificity'], 4),
        'Test_AUC':         round(v['AUC'],         4),
        'eval_set':         'Val Set',
        'BG_Vessel_ratio':  round(bg_px / max(fg_px, 1), 1),
        'epochs':           25,
        'model':            'IterNet',
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
        zip(v['history']['loss'], v['history']['val_dice']), 1
    ):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

print(f"\n최고 모델: {save_data['best_model']}")
print(f'평가 기준: {save_data["eval_set"]}  ({len(val_img_paths)}장)')
